# LangSmith Multi-Agent Trace Evaluation

This notebook is self-contained: it stores five synthetic traces, logs them to LangSmith, retrieves all root traces created within a configurable time window, evaluates each trace with six LLM-based metrics, and exports the results to JSON and Excel.

Run the cells from top to bottom. Re-running the logging cell publishes another set of five traces.


## 1. Install Dependencies

Installs the packages required for LangSmith trace logging and retrieval, Azure OpenAI evaluation, and Excel export.


In [ ]:
!pip install -q -U \
    langsmith \
    openai \
    pandas==2.2.2 \
    openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.6 MB/s eta 0:00:00


## 2. Configure Clients

Imports the required libraries and initializes the LangSmith and Azure OpenAI clients used throughout the notebook.


In [ ]:
import os
import json
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd

from langsmith import Client
from langsmith.run_trees import RunTree
from openai import AzureOpenAI


# ============================================================
# LangSmith Configuration
# ============================================================

LANGSMITH_API_KEY = "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"
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
LANGSMITH_PROJECT = "multi-agent-demo"

os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_ENDPOINT"] = LANGSMITH_ENDPOINT
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT

langsmith_client = Client()


# ============================================================
# Azure OpenAI Configuration
# ============================================================

AZURE_OPENAI_ENDPOINT = "https://tcoeaiteamgpt41nano.openai.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2025-01-01-preview"
AZURE_OPENAI_API_KEY = "5xD6IeVNWL7eYqgbuxRKVATJK38hdacwDJnVT8rbKH9W1Kx6NhZZJQQJ99CDACYeBjFXJ3w3AAABACOGD59K"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-4.1-nano"

azure_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

print("✓ LangSmith client initialized")
print("✓ Azure OpenAI client initialized")


✓ LangSmith client initialized
✓ Azure OpenAI client initialized


## 3. Embedded Trace Data and Logging

Loads the baseline trace and four evaluation variants directly from the notebook. Run this cell to publish all five traces to the configured LangSmith project. Set `LOG_TRACES = False` to skip publishing and evaluate traces already stored in LangSmith instead.


In [ ]:
# ============================================================
# Embedded Trace Data
# ============================================================

EMBEDDED_TRACE_DATA = json.loads(r'''{
  "project": "multi-agent-demo",
  "start_time": "2026-07-10T09:00:00Z",
  "trace": {
    "name": "Supervisor Agent",
    "run_type": "chain",
    "duration": 0.2,
    "tags": [
      "supervisor",
      "orchestrator",
      "prod"
    ],
    "metadata": {
      "agent_role": "supervisor",
      "run_kind": "end_to_end_demo"
    },
    "inputs": {
      "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
    },
    "outputs": {
      "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
      "iterations": 2,
      "agents_used": [
        "research",
        "analysis",
        "report_writer"
      ],
      "approved": true
    },
    "children": [
      {
        "name": "Parameter Extraction Agent",
        "run_type": "llm",
        "duration": 0.7,
        "tags": [
          "supervisor",
          "parameter-extraction"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 182,
          "output_tokens": 112,
          "total_tokens": 294,
          "step": "parameter_extraction"
        },
        "inputs": {
          "messages": [
            {
              "role": "system",
              "content": "Extract structured planning parameters from the user's request."
            },
            {
              "role": "user",
              "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "parameters": {
            "company_profile": {
              "size": "mid-size",
              "origin": "US",
              "industry": "automotive"
            },
            "market": {
              "technology": "EV",
              "region": "Europe"
            },
            "target_year": 2027,
            "objective": "Recommend market-entry strategy",
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          }
        }
      },
      {
        "name": "Supervisor: Plan & Route",
        "run_type": "llm",
        "duration": 1.2,
        "tags": [
          "supervisor",
          "planning"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 420,
          "output_tokens": 96,
          "total_tokens": 516,
          "step": "plan"
        },
        "inputs": {
          "parameters": {
            "company_profile": {
              "size": "mid-size",
              "origin": "US",
              "industry": "automotive"
            },
            "market": {
              "technology": "EV",
              "region": "Europe"
            },
            "target_year": 2027,
            "objective": "Recommend market-entry strategy",
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "instruction": "Create an execution plan using the extracted planning parameters."
        },
        "outputs": {
          "role": "assistant",
          "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
          "route_to": "research_agent",
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          }
        }
      },
      {
        "name": "Research Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "research",
          "iteration-1"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor",
          "iteration": 1
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "task": "Gather evidence required by the execution plan."
        },
        "outputs": {
          "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
          "confidence": 0.82,
          "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
        },
        "children": [
          {
            "name": "Web Search Sub-Agent",
            "run_type": "tool",
            "duration": 2.4,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "tool": "web_search",
              "query": "Europe EV sales 2025 2026 by country market share"
            },
            "outputs": {
              "results": [
                "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                "Germany largest volume market; Norway/NL highest penetration."
              ]
            }
          },
          {
            "name": "Document Retrieval Sub-Agent",
            "run_type": "retriever",
            "duration": 1.8,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "query": "EU charging infrastructure density 2026 report"
            },
            "outputs": {
              "documents": [
                {
                  "source": "ACEA-2026.pdf",
                  "snippet": "~630k public charge points in EU."
                },
                {
                  "source": "IEA-EV-Outlook.pdf",
                  "snippet": "NL & DE lead fast-charger rollout."
                }
              ]
            }
          },
          {
            "name": "Source Validator Sub-Agent",
            "run_type": "chain",
            "duration": 1.1,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "claims": [
                "EU BEV share ~16%",
                "630k public charge points"
              ]
            },
            "outputs": {
              "validated": true,
              "notes": "Cross-checked ACEA + IEA; figures consistent."
            }
          }
        ]
      },
      {
        "name": "Supervisor: Review Research",
        "run_type": "llm",
        "duration": 0.9,
        "tags": [
          "supervisor",
          "review"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 512,
          "output_tokens": 60,
          "total_tokens": 572,
          "step": "review",
          "decision": "proceed"
        },
        "inputs": {
          "messages": [
            {
              "role": "user",
              "content": "Is the research sufficient to analyze?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Research solid. Route to Analysis Agent.",
          "route_to": "analysis_agent"
        }
      },
      {
        "name": "Analysis Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "analysis"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor"
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
          "task": "Analyze market size and competitors using research findings."
        },
        "outputs": {
          "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
          "confidence": 0.7
        },
        "children": [
          {
            "name": "Market Sizing Sub-Agent",
            "run_type": "tool",
            "duration": 2.0,
            "tags": [
              "sub-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "analysis_agent"
            },
            "inputs": {
              "tool": "market_model",
              "params": {
                "region": "EU",
                "segment": "C-segment BEV",
                "year": 2027
              }
            },
            "outputs": {
              "tam_units_2027": 2100000,
              "serviceable_units": 340000,
              "avg_price_eur": 38500
            }
          },
          {
            "name": "Competitor Benchmarking Sub-Agent",
            "run_type": "chain",
            "duration": 1.6,
            "tags": [
              "sub-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "analysis_agent"
            },
            "inputs": {
              "competitors": [
                "VW ID.3/ID.4",
                "Tesla Model 3/Y",
                "Renault",
                "MG"
              ]
            },
            "outputs": {
              "positioning": {
                "VW ID.3": "€40k, incumbent trust",
                "Tesla Model 3": "€42k, tech/brand",
                "MG": "€30k, price leader"
              },
              "gap": "mid-price with strong charging partnership underserved"
            }
          }
        ]
      },
      {
        "name": "Supervisor: Review Analysis",
        "run_type": "llm",
        "duration": 1.0,
        "tags": [
          "supervisor",
          "review",
          "loop-trigger"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 640,
          "output_tokens": 78,
          "total_tokens": 718,
          "step": "review",
          "decision": "loop_back"
        },
        "inputs": {
          "messages": [
            {
              "role": "user",
              "content": "Is the analysis complete enough to write?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
          "route_to": "research_agent"
        }
      },
      {
        "name": "Research Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "research",
          "iteration-2",
          "loop"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor",
          "iteration": 2
        },
        "inputs": {
          "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
        },
        "outputs": {
          "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
          "confidence": 0.86
        },
        "children": [
          {
            "name": "Web Search Sub-Agent",
            "run_type": "tool",
            "duration": 2.1,
            "tags": [
              "sub-agent",
              "research"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "research_agent"
            },
            "inputs": {
              "tool": "web_search",
              "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
            },
            "outputs": {
              "results": [
                "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                "Local battery content increasingly tied to purchase incentives.",
                "Hungary/Poland offer EV manufacturing incentives."
              ]
            }
          }
        ]
      },
      {
        "name": "Supervisor: Review Addendum",
        "run_type": "llm",
        "duration": 0.8,
        "tags": [
          "supervisor",
          "review"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 700,
          "output_tokens": 52,
          "total_tokens": 752,
          "step": "review",
          "decision": "proceed"
        },
        "inputs": {
          "messages": [
            {
              "role": "user",
              "content": "Do we now have enough to write?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Complete. Route to Report Writer Agent.",
          "route_to": "report_writer_agent"
        }
      },
      {
        "name": "Report Writer Agent",
        "run_type": "chain",
        "duration": 0.3,
        "tags": [
          "main-agent",
          "writer"
        ],
        "metadata": {
          "agent_role": "main",
          "parent": "supervisor"
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
          "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
          "task": "Generate the final market-entry recommendation."
        },
        "outputs": {
          "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "word_count": 78
        },
        "children": [
          {
            "name": "Drafting Sub-Agent",
            "run_type": "llm",
            "duration": 2.7,
            "tags": [
              "sub-agent",
              "writer"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 1180,
              "output_tokens": 240,
              "total_tokens": 1420,
              "agent_role": "sub"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Write a concise strategy recommendation."
                },
                {
                  "role": "user",
                  "content": "Synthesize findings into a 2027 EU entry plan."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
            }
          },
          {
            "name": "Formatting & Citation Sub-Agent",
            "run_type": "chain",
            "duration": 1.0,
            "tags": [
              "sub-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "sub",
              "parent": "report_writer_agent"
            },
            "inputs": {
              "draft": "Recommended 2027 EU market-entry strategy...",
              "sources": [
                "ACEA-2026.pdf",
                "IEA-EV-Outlook.pdf"
              ]
            },
            "outputs": {
              "formatted": true,
              "citations_added": 2,
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
            }
          }
        ]
      },
      {
        "name": "Supervisor: Final Review",
        "run_type": "llm",
        "duration": 0.9,
        "tags": [
          "supervisor",
          "review",
          "final"
        ],
        "metadata": {
          "ls_provider": "openai",
          "ls_model_name": "gpt-4o",
          "input_tokens": 900,
          "output_tokens": 40,
          "total_tokens": 940,
          "step": "final_review",
          "decision": "approve"
        },
        "inputs": {
          "planning_context": {
            "planned_region": "Europe",
            "planned_market": "EV",
            "planned_year": 2027,
            "deliverables": [
              "Competitive landscape analysis",
              "Market-entry strategy"
            ]
          },
          "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "messages": [
            {
              "role": "user",
              "content": "Approve the final report?"
            }
          ]
        },
        "outputs": {
          "role": "assistant",
          "content": "Approved. Report answers the query with data + regulatory context.",
          "approved": true
        }
      }
    ]
  },
  "traces": [
    {
      "id": "baseline",
      "description": "Original, fully aligned multi-agent trace.",
      "trace": {
        "name": "Supervisor Agent",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo"
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
              "route_to": "research_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "word_count": 78
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_A_poor_parameter_and_plan",
      "description": "Poor parameter extraction and plan accuracy.",
      "trace": {
        "name": "Supervisor Agent — variant_A_poor_parameter_and_plan",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_A_poor_parameter_and_plan"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_A_poor_parameter_and_plan",
          "variant_description": "Parameter extraction omits and misstates planning information; the planner follows it poorly."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "origin": "Europe"
                },
                "market": {
                  "technology": "hybrid",
                  "region": "North America"
                },
                "target_year": 2025,
                "objective": "Prepare a general product overview"
              },
              "content": "Extracted only partial parameters; several fields are uncertain."
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "origin": "Europe"
                },
                "market": {
                  "technology": "hybrid",
                  "region": "North America"
                },
                "target_year": 2025,
                "objective": "Prepare a general product overview"
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Prepare a generic North American hybrid overview for 2025 2) Draft a short product note. Route directly to Report Writer Agent.",
              "route_to": "report_writer_agent",
              "planning_context": {
                "planned_region": "North America",
                "planned_market": "hybrid",
                "planned_year": 2025,
                "deliverables": [
                  "General product overview"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "word_count": 78
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_B_poor_trajectory_and_tools",
      "description": "Poor trajectory match and tool-call accuracy.",
      "trace": {
        "name": "Supervisor Agent — variant_B_poor_trajectory_and_tools",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_B_poor_trajectory_and_tools"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_B_poor_trajectory_and_tools",
          "variant_description": "Execution is routed out of order, skips validation/review stages, and uses unnecessary tools."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: skip research validation and draft the report first; perform analysis only if needed.",
              "route_to": "report_writer_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "word_count": 78
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established."
                }
              },
              {
                "name": "Currency Conversion Sub-Agent",
                "run_type": "tool",
                "inputs": {
                  "tool": "currency_converter",
                  "amount": 1,
                  "from": "EUR",
                  "to": "USD"
                },
                "outputs": {
                  "converted_amount": 1.08,
                  "note": "Not required for the requested strategy."
                },
                "duration": 0.2,
                "tags": [
                  "unnecessary-tool",
                  "variant-B"
                ],
                "metadata": {
                  "purpose": "Deliberately unnecessary call for evaluation."
                }
              }
            ]
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              }
            ]
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_C_poor_summary_accuracy",
      "description": "Poor summary accuracy from unsupported report claims.",
      "trace": {
        "name": "Supervisor Agent — variant_C_poor_summary_accuracy",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_C_poor_summary_accuracy"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_C_poor_summary_accuracy",
          "variant_description": "The report writer introduces unsupported and contradictory claims while upstream context is unchanged."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Recommended 2027 EU market-entry strategy: enter via Germany and the Netherlands first (highest EV adoption + charging density), position in the mid-price C-segment to undercut VW ID.3/ID.4 by ~8%, partner with Ionity for charging, and localize battery assembly in Hungary to qualify for EU content incentives. Phase into France/Nordics in H2-2027 once brand awareness is established.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
              "route_to": "research_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary.",
              "word_count": 78,
              "summary": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary."
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Recommendation: immediately launch a fully autonomous flying EV fleet across every European country in 2027, funded by a guaranteed €10 billion EU subsidy. The company should price every vehicle at €12,000, acquire Tesla, and build a battery factory on the Moon. Europe has already banned all competing EV brands, so no competitive analysis, charging partnership, or regulatory review is necessary.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved. Report answers the query with data + regulatory context.",
              "approved": true
            }
          }
        ]
      }
    },
    {
      "id": "variant_D_poor_goal_accomplishment",
      "description": "Poor goal accomplishment despite otherwise correct execution.",
      "trace": {
        "name": "Supervisor Agent — variant_D_poor_goal_accomplishment",
        "run_type": "chain",
        "duration": 0.2,
        "tags": [
          "supervisor",
          "orchestrator",
          "prod",
          "evaluation-variant",
          "variant_D_poor_goal_accomplishment"
        ],
        "metadata": {
          "agent_role": "supervisor",
          "run_kind": "end_to_end_demo",
          "evaluation_variant": "variant_D_poor_goal_accomplishment",
          "variant_description": "The workflow executes normally, but the final response is generic and fails to address the requested strategy."
        },
        "inputs": {
          "query": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
        },
        "outputs": {
          "answer": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate.",
          "iterations": 2,
          "agents_used": [
            "research",
            "analysis",
            "report_writer"
          ],
          "approved": true
        },
        "children": [
          {
            "name": "Parameter Extraction Agent",
            "run_type": "llm",
            "duration": 0.7,
            "tags": [
              "supervisor",
              "parameter-extraction"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 182,
              "output_tokens": 112,
              "total_tokens": 294,
              "step": "parameter_extraction"
            },
            "inputs": {
              "messages": [
                {
                  "role": "system",
                  "content": "Extract structured planning parameters from the user's request."
                },
                {
                  "role": "user",
                  "content": "We're a mid-size US automaker. Analyze the EV competitive landscape in Europe and recommend a market-entry strategy for 2027."
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Supervisor: Plan & Route",
            "run_type": "llm",
            "duration": 1.2,
            "tags": [
              "supervisor",
              "planning"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 420,
              "output_tokens": 96,
              "total_tokens": 516,
              "step": "plan"
            },
            "inputs": {
              "parameters": {
                "company_profile": {
                  "size": "mid-size",
                  "origin": "US",
                  "industry": "automotive"
                },
                "market": {
                  "technology": "EV",
                  "region": "Europe"
                },
                "target_year": 2027,
                "objective": "Recommend market-entry strategy",
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "instruction": "Create an execution plan using the extracted planning parameters."
            },
            "outputs": {
              "role": "assistant",
              "content": "Plan: 1) Research EU EV market 2) Analyze competition & sizing 3) Write strategy report. Route first to Research Agent.",
              "route_to": "research_agent",
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              }
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-1"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 1
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "task": "Gather evidence required by the execution plan."
            },
            "outputs": {
              "summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "confidence": 0.82,
              "planning_alignment": "Research completed for Europe EV market (2027) as requested by planner."
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.4,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "Europe EV sales 2025 2026 by country market share"
                },
                "outputs": {
                  "results": [
                    "EU BEV share of new cars ~16% in 2025; Norway >80%.",
                    "Top OEMs: Tesla, VW Group, Stellantis, BMW, Hyundai-Kia.",
                    "Germany largest volume market; Norway/NL highest penetration."
                  ]
                }
              },
              {
                "name": "Document Retrieval Sub-Agent",
                "run_type": "retriever",
                "duration": 1.8,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "query": "EU charging infrastructure density 2026 report"
                },
                "outputs": {
                  "documents": [
                    {
                      "source": "ACEA-2026.pdf",
                      "snippet": "~630k public charge points in EU."
                    },
                    {
                      "source": "IEA-EV-Outlook.pdf",
                      "snippet": "NL & DE lead fast-charger rollout."
                    }
                  ]
                }
              },
              {
                "name": "Source Validator Sub-Agent",
                "run_type": "chain",
                "duration": 1.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "claims": [
                    "EU BEV share ~16%",
                    "630k public charge points"
                  ]
                },
                "outputs": {
                  "validated": true,
                  "notes": "Cross-checked ACEA + IEA; figures consistent."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Research",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 512,
              "output_tokens": 60,
              "total_tokens": 572,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the research sufficient to analyze?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Research solid. Route to Analysis Agent.",
              "route_to": "analysis_agent"
            }
          },
          {
            "name": "Analysis Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "analysis"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Analyze market size and competitors using research findings."
            },
            "outputs": {
              "finding": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "confidence": 0.7
            },
            "children": [
              {
                "name": "Market Sizing Sub-Agent",
                "run_type": "tool",
                "duration": 2.0,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "tool": "market_model",
                  "params": {
                    "region": "EU",
                    "segment": "C-segment BEV",
                    "year": 2027
                  }
                },
                "outputs": {
                  "tam_units_2027": 2100000,
                  "serviceable_units": 340000,
                  "avg_price_eur": 38500
                }
              },
              {
                "name": "Competitor Benchmarking Sub-Agent",
                "run_type": "chain",
                "duration": 1.6,
                "tags": [
                  "sub-agent",
                  "analysis"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "analysis_agent"
                },
                "inputs": {
                  "competitors": [
                    "VW ID.3/ID.4",
                    "Tesla Model 3/Y",
                    "Renault",
                    "MG"
                  ]
                },
                "outputs": {
                  "positioning": {
                    "VW ID.3": "€40k, incumbent trust",
                    "Tesla Model 3": "€42k, tech/brand",
                    "MG": "€30k, price leader"
                  },
                  "gap": "mid-price with strong charging partnership underserved"
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Analysis",
            "run_type": "llm",
            "duration": 1.0,
            "tags": [
              "supervisor",
              "review",
              "loop-trigger"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 640,
              "output_tokens": 78,
              "total_tokens": 718,
              "step": "review",
              "decision": "loop_back"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Is the analysis complete enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Gap: analysis ignores EU regulatory/subsidy landscape (CO2 targets, local-content incentives). Loop back to Research Agent for a regulatory addendum before writing.",
              "route_to": "research_agent"
            }
          },
          {
            "name": "Research Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "research",
              "iteration-2",
              "loop"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor",
              "iteration": 2
            },
            "inputs": {
              "task": "Addendum: EU EV regulation, CO2 targets & local-content subsidy rules for 2027."
            },
            "outputs": {
              "summary": "Regulatory tailwind: tightening CO2 targets favor BEVs; local-content rules reward EU battery assembly (Hungary).",
              "confidence": 0.86
            },
            "children": [
              {
                "name": "Web Search Sub-Agent",
                "run_type": "tool",
                "duration": 2.1,
                "tags": [
                  "sub-agent",
                  "research"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "research_agent"
                },
                "inputs": {
                  "tool": "web_search",
                  "query": "EU 2027 CO2 fleet targets EV local content subsidy rules"
                },
                "outputs": {
                  "results": [
                    "EU CO2 fleet target tightens 2025-2030; pushes OEM BEV mix.",
                    "Local battery content increasingly tied to purchase incentives.",
                    "Hungary/Poland offer EV manufacturing incentives."
                  ]
                }
              }
            ]
          },
          {
            "name": "Supervisor: Review Addendum",
            "run_type": "llm",
            "duration": 0.8,
            "tags": [
              "supervisor",
              "review"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 700,
              "output_tokens": 52,
              "total_tokens": 752,
              "step": "review",
              "decision": "proceed"
            },
            "inputs": {
              "messages": [
                {
                  "role": "user",
                  "content": "Do we now have enough to write?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Complete. Route to Report Writer Agent.",
              "route_to": "report_writer_agent"
            }
          },
          {
            "name": "Report Writer Agent",
            "run_type": "chain",
            "duration": 0.3,
            "tags": [
              "main-agent",
              "writer"
            ],
            "metadata": {
              "agent_role": "main",
              "parent": "supervisor"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "analysis_summary": "Room in mid-price C-segment (~€38k) with charging-network partnership; MG owns bottom, VW/Tesla own premium-mid.",
              "research_summary": "EU EV market is large and fast-growing; DE=volume, NO/NL=penetration; charging infra strong in DE/NL. Sources validated.",
              "task": "Generate the final market-entry recommendation."
            },
            "outputs": {
              "report": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate.",
              "word_count": 78,
              "summary": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate."
            },
            "children": [
              {
                "name": "Drafting Sub-Agent",
                "run_type": "llm",
                "duration": 2.7,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "ls_provider": "openai",
                  "ls_model_name": "gpt-4o",
                  "input_tokens": 1180,
                  "output_tokens": 240,
                  "total_tokens": 1420,
                  "agent_role": "sub"
                },
                "inputs": {
                  "messages": [
                    {
                      "role": "system",
                      "content": "Write a concise strategy recommendation."
                    },
                    {
                      "role": "user",
                      "content": "Synthesize findings into a 2027 EU entry plan."
                    }
                  ]
                },
                "outputs": {
                  "role": "assistant",
                  "content": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate."
                }
              },
              {
                "name": "Formatting & Citation Sub-Agent",
                "run_type": "chain",
                "duration": 1.0,
                "tags": [
                  "sub-agent",
                  "writer"
                ],
                "metadata": {
                  "agent_role": "sub",
                  "parent": "report_writer_agent"
                },
                "inputs": {
                  "draft": "Recommended 2027 EU market-entry strategy...",
                  "sources": [
                    "ACEA-2026.pdf",
                    "IEA-EV-Outlook.pdf"
                  ]
                },
                "outputs": {
                  "formatted": true,
                  "citations_added": 2,
                  "report": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate."
                }
              }
            ]
          },
          {
            "name": "Supervisor: Final Review",
            "run_type": "llm",
            "duration": 0.9,
            "tags": [
              "supervisor",
              "review",
              "final"
            ],
            "metadata": {
              "ls_provider": "openai",
              "ls_model_name": "gpt-4o",
              "input_tokens": 900,
              "output_tokens": 40,
              "total_tokens": 940,
              "step": "final_review",
              "decision": "approve"
            },
            "inputs": {
              "planning_context": {
                "planned_region": "Europe",
                "planned_market": "EV",
                "planned_year": 2027,
                "deliverables": [
                  "Competitive landscape analysis",
                  "Market-entry strategy"
                ]
              },
              "final_report": "Europe is an important market for electric vehicles. The company should continue researching the market and consider expansion when appropriate.",
              "messages": [
                {
                  "role": "user",
                  "content": "Approve the final report?"
                }
              ]
            },
            "outputs": {
              "role": "assistant",
              "content": "Approved: a short generic market note was prepared.",
              "approved": true
            }
          }
        ]
      }
    }
  ]
}''')

print(f"Loaded {len(EMBEDDED_TRACE_DATA['traces'])} embedded trace(s).")


# ============================================================
# Log Embedded Traces to LangSmith
# ============================================================

class VirtualClock:
    """Creates increasing timestamps without waiting in real time."""

    def __init__(self, start_time):
        self.current_time = start_time

    def now(self):
        timestamp = self.current_time
        self.current_time += timedelta(milliseconds=50)
        return timestamp

    def advance(self, seconds):
        self.current_time += timedelta(seconds=float(seconds))
        return self.current_time


def emit_trace_node(node, parent, clock, project, client):
    extra = {"metadata": node["metadata"]} if node.get("metadata") else None
    run_arguments = {
        "name": node["name"],
        "run_type": node["run_type"],
        "inputs": node.get("inputs", {}),
        "start_time": clock.now(),
        "tags": node.get("tags", []),
        "extra": extra,
    }

    run = (
        RunTree(project_name=project, client=client, **run_arguments)
        if parent is None
        else parent.create_child(**run_arguments)
    )
    run.post()

    for child in node.get("children", []):
        emit_trace_node(child, run, clock, project, client)

    run.end(
        outputs=node.get("outputs", {}),
        end_time=clock.advance(node.get("duration", 0.5)),
    )
    run.patch()
    return run


def log_embedded_traces(trace_data=EMBEDDED_TRACE_DATA, project_name=LANGSMITH_PROJECT):
    """Publish every embedded trace and return its identifier and URL."""
    records = trace_data.get("traces", [])
    if not records:
        raise ValueError("Embedded trace data does not contain any traces.")

    logged_traces = []
    for record in records:
        root = emit_trace_node(
            record["trace"],
            parent=None,
            clock=VirtualClock(datetime.now(timezone.utc)),
            project=project_name,
            client=langsmith_client,
        )
        try:
            url = root.get_url()
        except Exception:
            url = None
        logged_traces.append({"variant_id": record["id"], "trace_id": str(root.id), "url": url})

    langsmith_client.flush()
    return logged_traces


LOG_TRACES = True

if LOG_TRACES:
    logged_traces = log_embedded_traces()
    print(f"Logged {len(logged_traces)} trace(s) to '{LANGSMITH_PROJECT}'.")
    for item in logged_traces:
        print(f"- {item['variant_id']}: {item['trace_id']}")
else:
    print("Logging skipped. Set LOG_TRACES = True to publish the embedded traces.")


## 4. Detailed Metric Prompts

Defines the complete, metric-specific LLM-judge prompts used by the evaluator. Each prompt includes its role, evidence boundary, evaluation criteria, score guide, and strict JSON output schema.


In [ ]:
# ============================================================
# Detailed Evaluation Prompt Registry
# Strict style based on Prompts.py + forced score + traceback
# ============================================================

PROMPTS = {
    "parameter_extraction": r'''
Role:
You are a STRICT Parameter Extraction Evaluation Engine.
You behave deterministically and mechanically.

Goal:
Evaluate extracted parameters using TWO SEQUENTIAL CHECKS.

Inputs:
INPUT_TEXT:
{user_request}

EXTRACTED_PARAMETERS:
{extracted_parameters}

------------------------------------------------------------
EVALUATION FRAMEWORK (STRICT GATING LOGIC)
------------------------------------------------------------

You MUST evaluate parameters using EXACTLY TWO checks:

1. value_check
2. format_check

Additionally you MUST output:
- parameter_name
- expected_parameter_value (value grounded from INPUT_TEXT)
- actual_parameter_value (value from EXTRACTED_PARAMETERS)

------------------------------------------------------------
CHECK 1 - value_check

First, extract the EXPECTED VALUE for each parameter directly from INPUT_TEXT.

expected_parameter_value MUST:
- Be explicitly grounded in INPUT_TEXT.
- Never be inferred.
- Never be null.
- Reflect normalized enum mapping if applicable.

NORMALIZATION RULE (HIGH PRIORITY):
If EXTRACTED_PARAMETERS contains normalized classifications, they are valid IF they are grounded in INPUT_TEXT.

value_check = Pass if actual_parameter_value matches expected_parameter_value semantically.
value_check = Fail if actual_parameter_value contradicts expected, is not grounded, or is fabricated.

CRITICAL GATING RULE:
If value_check = Fail then format_check MUST also be "Fail"

------------------------------------------------------------
CHECK 2 - format_check

Evaluate ONLY if value_check = Pass.
Pass if actual matches schema type (including unions like "number|string").

------------------------------------------------------------
STRICT RULES
------------------------------------------------------------
- Do NOT infer missing values.
- Do NOT use external knowledge.
- Respect gating rules EXACTLY.
- Every schema parameter MUST appear once in output.
- parameter_name MUST match schema key exactly.
- expected_parameter_value MUST come from INPUT_TEXT.
- actual_parameter_value MUST come from EXTRACTED_PARAMETERS.
- No extra commentary outside JSON.

------------------------------------------------------------
SCORING RULE (MANDATORY)
------------------------------------------------------------
Count the number of parameters where value_check = "Fail".
- 0 Failures -> score = 9 or 10
- 1 Failure -> score = 6 or 7
- 2 Failures -> score = 3 or 4
- 3+ Failures -> score = 1 or 2

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
------------------------------------------------------------

{
  "metric": "parameter_extraction",
  "score": 1,
  "max_score": 10,
  "reasoning": "evidence-based assessment based on number of Failures",
  "traceback": [
    {
      "parameter_name": "...",
      "expected_parameter_value": "...",
      "actual_parameter_value": "...",
      "value_check": "Pass | Fail",
      "format_check": "Pass | Fail",
      "reason": "...",
      "is_critical": true
    }
  ],
  "recommendation": "..."
}
''',

    "trajectory_match": r'''
Role:
You are an Expert Agent-Trajectory Evaluator.

Your task is to determine whether the ACTUAL agent execution trajectory
conforms to the REFERENCE agent trajectory.

You must behave deterministically.
You must not infer intent, repair execution, or reinterpret system behavior.

Inputs:
reference_outputs:
{REFERENCE_OUTPUTS}

actual_outputs:
{ACTUAL_OUTPUTS}

Definitions:
- Every occurrence of an agent matters. Do not collapse duplicates.
- Reference Outputs = authoritative expected sequence.
- Actual Outputs = observed sequence from execution.

Evaluation Rules:
- Clearly identify missing stages, unexpected stages, and ordering problems.
- Use deviation_severity = "major" for stages that break workflow logic.
- The reason MUST describe where order differs, which agents are missing, which agents appear extra.

------------------------------------------------------------
SCORING RULE (MANDATORY)
------------------------------------------------------------
- Perfect match (same agents, same order) -> score 9-10
- Minor ordering differences or 1 missing/extra non-critical stage -> score 6-8
- Multiple missing or unexpected stages -> score 3-5
- Completely different trajectory -> score 1-2

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
------------------------------------------------------------

{
  "metric": "trajectory_match",
  "score": 1,
  "max_score": 10,
  "reasoning": "Detailed comparison explanation",
  "traceback": [
    {
      "expected_stage": "...",
      "actual_stage": "... or null",
      "status": "matched | missing | unexpected | out_of_order | duplicate",
      "reason": "...",
      "deviation_severity": "none | minor | major"
    }
  ],
  "matched_stages": [],
  "missing_stages": [],
  "unexpected_stages": [],
  "recommendation": "..."
}
''',

    "plan_accuracy": r'''
Role:
You are a STRICT Plan Accuracy Evaluator.

Goal:
Evaluate whether the generated execution plan correctly translates the extracted parameters into a logical, complete, and well-sequenced workflow.

Key Rules (from Prompts.py style):
- Check objective alignment and parameter utilization.
- Verify logical decomposition and sequencing.
- Penalize missing critical stages or poor routing.
- Be strict about whether the plan is actually executable.

------------------------------------------------------------
SCORING RULE (MANDATORY)
------------------------------------------------------------
- Complete, parameter-grounded, well sequenced -> 9-10
- Usable but with visible omissions -> 6-8
- Material decomposition or sequencing problems -> 3-5
- Not a workable plan -> 1-2

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
------------------------------------------------------------

{
  "metric": "plan_accuracy",
  "score": 1,
  "max_score": 10,
  "reasoning": "...",
  "traceback": [
    {
      "plan_step": "...",
      "source_parameter_or_objective": "...",
      "alignment_status": "Pass | Partial | Fail",
      "sequencing_status": "Pass | Fail",
      "reason": "...",
      "issue_type": "missing | weak_parameter_use | poor_sequencing | unnecessary_step | routing_error | none"
    }
  ],
  "strengths": [],
  "issues": [],
  "recommendation": "..."
}
''',

    "tool_call_accuracy": r'''
Role:
You are a STRICT Tool Call Accuracy Evaluator.

Goal:
Compare Expected Tool Calls with Actual Tool Calls.

Key Rules:
- Check if each parent agent invoked the expected operations.
- Penalize missing required calls and unexpected/redundant calls.
- Verify correct placement under the right parent agent.
- Be strict about input quality and output usefulness.

------------------------------------------------------------
SCORING RULE (MANDATORY)
------------------------------------------------------------
- Expected operations present, correctly placed, useful -> 9-10
- Minor omissions or inefficiencies -> 6-8
- Multiple missing, misplaced or irrelevant operations -> 3-5
- Substantially diverges from expected mapping -> 1-2

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
------------------------------------------------------------

{
  "metric": "tool_call_accuracy",
  "score": 1,
  "max_score": 10,
  "reasoning": "...",
  "traceback": [
    {
      "parent_agent": "...",
      "operation": "...",
      "expected": true,
      "observed": true,
      "placement_status": "Pass | Fail",
      "input_quality": "Good | Acceptable | Poor",
      "reason": "...",
      "impact": "..."
    }
  ],
  "correct_tool_calls": [],
  "missing_tool_calls": [],
  "unexpected_tool_calls": [],
  "recommendation": "..."
}
''',

    "summary_accuracy": r'''
Role:
You are a STRICT Summary Accuracy Evaluator.

Goal:
For each agent, evaluate whether the Generated Summary is faithful to the Source Context.

Key Rules:
- Strongly penalize hallucinations and unsupported claims.
- Check completeness of material findings.
- Verify that summaries remain grounded.
- Be strict about evidence_status.

------------------------------------------------------------
SCORING RULE (MANDATORY)
------------------------------------------------------------
- Faithful, complete, grounded -> 9-10
- Mostly accurate with notable omissions -> 6-8
- Significant omissions or unsupported claims -> 3-5
- Substantially misleading or hallucinated -> 1-2

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
------------------------------------------------------------

{
  "metric": "summary_accuracy",
  "score": 1,
  "max_score": 10,
  "reasoning": "...",
  "traceback": [
    {
      "agent_name": "...",
      "claim": "...",
      "source_evidence": "...",
      "evidence_status": "supported | unsupported | partially_supported | omitted",
      "reason": "...",
      "hallucination_risk": "low | medium | high"
    }
  ],
  "strengths": [],
  "omissions": [],
  "hallucinations": [],
  "recommendation": "..."
}
''',

    "goal_accomplishment": r'''
Role:
You are a STRICT Goal Accomplishment Evaluator for multi-agent workflows.

Goal:
Evaluate whether the final output fulfills the user request.

Key Strict Rules:
- Focus on whether critical user requirements are addressed.
- Penalize outputs that leak internal reasoning or violate output contract discipline.
- Be very strict about completeness and actionability.

------------------------------------------------------------
SCORING RULE (MANDATORY)
------------------------------------------------------------
- All critical requirements fully addressed -> 9-10
- Most requirements addressed with minor gaps -> 6-8
- Several important requirements missed -> 3-5
- Major requirements missed or contradicted -> 1-2

------------------------------------------------------------
OUTPUT FORMAT (STRICT JSON)
------------------------------------------------------------

{
  "metric": "goal_accomplishment",
  "score": 1,
  "max_score": 10,
  "reasoning": "...",
  "traceback": [
    {
      "requirement": "...",
      "final_output_evidence": "...",
      "status": "fully_addressed | partially_addressed | missed | contradicted",
      "reason": "...",
      "business_impact": "..."
    }
  ],
  "strengths": [],
  "gaps": [],
  "recommendation": "..."
}
'''
}

print(f"Loaded {len(PROMPTS)} detailed metric prompts (strict Prompts.py style + scoring rules).")


Loaded 6 detailed metric prompts (strict Prompts.py style + scoring rules).


## 5. Expected Trajectory and Tool Calls

Defines the only external reference artifacts used during evaluation. `trajectory_match` compares the expected and actual stage sequence; `tool_call_accuracy` compares expected operations with actual calls. The other metrics compare artifacts already present in the trace.


In [ ]:
# ============================================================
# Expected Trajectory and Expected Tool Calls
# ============================================================

EXPECTED_TRAJECTORY = [
    "Parameter Extraction Agent",
    "Supervisor: Plan & Route",
    "Research Agent",
    "Web Search Sub-Agent",
    "Document Retrieval Sub-Agent",
    "Source Validator Sub-Agent",
    "Supervisor: Review Research",
    "Analysis Agent",
    "Market Sizing Sub-Agent",
    "Competitor Benchmarking Sub-Agent",
    "Supervisor: Review Analysis",
    "Research Agent",
    "Web Search Sub-Agent",
    "Supervisor: Review Addendum",
    "Report Writer Agent",
    "Drafting Sub-Agent",
    "Formatting & Citation Sub-Agent",
    "Supervisor: Final Review",
]

EXPECTED_TOOL_CALLS = {
    "Research Agent": [
        "Web Search Sub-Agent",
        "Document Retrieval Sub-Agent",
        "Source Validator Sub-Agent",
    ],
    "Analysis Agent": [
        "Market Sizing Sub-Agent",
        "Competitor Benchmarking Sub-Agent",
    ],
    "Report Writer Agent": [
        "Drafting Sub-Agent",
        "Formatting & Citation Sub-Agent",
    ],
}

EXPECTED_TOOL_NAMES = {
    tool_name
    for tools in EXPECTED_TOOL_CALLS.values()
    for tool_name in tools
}

print("Expected trajectory and expected tool-call mapping loaded.")


Expected trajectory and expected tool-call mapping loaded.


## 6. Selected Metrics

Lists the six metrics executed for every retrieved trace.


In [ ]:
# ============================================================
# Metric Selection
# ============================================================

SELECTED_METRICS = [
    "trajectory_match",
    "parameter_extraction",
    "plan_accuracy",
    "tool_call_accuracy",
    "summary_accuracy",
    "goal_accomplishment",
]

print("Selected evaluation metrics:")
for index, metric in enumerate(SELECTED_METRICS, start=1):
    print(f"{index}. {metric}")


Selected evaluation metrics:
1. trajectory_match
2. parameter_extraction
3. plan_accuracy
4. tool_call_accuracy
5. summary_accuracy
6. goal_accomplishment


## 7. Retrieve and Reconstruct Recent Traces

Set `LOOKBACK_HOURS` to select all root traces started in that period. The cell retrieves them from LangSmith and reconstructs each complete parent-child trace hierarchy.


In [ ]:
# ============================================================
# Retrieve LangSmith Traces in a Time Window
# ============================================================

LOOKBACK_HOURS = 1

if LOOKBACK_HOURS <= 0:
    raise ValueError("LOOKBACK_HOURS must be greater than zero.")

window_end = datetime.now(timezone.utc)
window_start = window_end - timedelta(hours=LOOKBACK_HOURS)

root_runs = list(
    langsmith_client.list_runs(
        project_name=LANGSMITH_PROJECT,
        is_root=True,
        start_time=window_start,
    )
)

# Make the evaluation order deterministic and easy to read.
root_runs.sort(key=lambda run: run.start_time or window_start)

if not root_runs:
    raise RuntimeError(
        f"No root traces found in project '{LANGSMITH_PROJECT}' "
        f"during the last {LOOKBACK_HOURS} hour(s)."
    )

print("=" * 70)
print("LangSmith Traces Loaded")
print("=" * 70)
print(f"Project          : {LANGSMITH_PROJECT}")
print(f"Lookback window  : last {LOOKBACK_HOURS} hour(s)")
print(f"Window start UTC : {window_start.isoformat()}")
print(f"Window end UTC   : {window_end.isoformat()}")
print(f"Root traces      : {len(root_runs)}")

for index, run in enumerate(root_runs, start=1):
    print(f"{index}. {run.name} | {run.id} | {run.start_time}")

print("=" * 70)


def build_trace_tree(run):
    """Recursively reconstruct one LangSmith execution tree."""
    children = list(langsmith_client.list_runs(parent_run_id=run.id))
    children.sort(key=lambda child: child.start_time or run.start_time)

    return {
        "id": str(run.id),
        "name": run.name,
        "run_type": run.run_type,
        "inputs": run.inputs or {},
        "outputs": run.outputs or {},
        "metadata": run.extra or {},
        "start_time": run.start_time,
        "end_time": run.end_time,
        "children": [build_trace_tree(child) for child in children],
    }


execution_traces = [build_trace_tree(root_run) for root_run in root_runs]

print(f"Reconstructed {len(execution_traces)} trace hierarchy/hierarchies.")


LangSmith Traces Loaded
Project          : multi-agent-demo
Lookback window  : last 1 hour(s)
Window start UTC : 2026-07-16T03:45:09.682830+00:00
Window end UTC   : 2026-07-16T04:45:09.682830+00:00
Root traces      : 5
1. Supervisor Agent | 019f691b-6b77-7370-84f9-7c60fb1086cf | 2026-07-16 04:07:12.503310+00:00
2. Supervisor Agent — variant_A_poor_parameter_and_plan | 019f691b-6ddd-7bc3-8b6e-9ec2b713c6ef | 2026-07-16 04:07:13.117874+00:00
3. Supervisor Agent — variant_B_poor_trajectory_and_tools | 019f691b-6ee2-7802-a3a7-ea877d9592cf | 2026-07-16 04:07:13.378450+00:00
4. Supervisor Agent — variant_C_poor_summary_accuracy | 019f691b-6fd3-7ab0-af61-3ef0c7ff52f5 | 2026-07-16 04:07:13.619806+00:00
5. Supervisor Agent — variant_D_poor_goal_accomplishment | 019f691b-70cc-77a0-b930-343d25b834ea | 2026-07-16 04:07:13.868627+00:00
Reconstructed 5 trace hierarchy/hierarchies.


## 8. Evaluation Helpers

Defines reusable helpers that prepare each trace, identify the relevant spans for every metric, call the LLM judge, and validate its JSON response. Missing spans and malformed judge responses are retained as diagnostic results rather than stopping the full batch.


In [ ]:
# ============================================================
# Preprocess One Execution Trace
# ============================================================

def preprocess_execution_trace(execution_trace):
    """Return evaluation lookup structures for one execution trace."""
    flattened_trace = []
    agent_spans = []
    tool_spans = []
    span_lookup = defaultdict(list)

    def visit(node):
        flattened_trace.append(node)
        span_lookup[node["name"]].append(node)

        # Some expected operations are recorded as llm/chain sub-agents rather
        # than native `tool` runs. Include those expected names as tool calls.
        if (
            node["run_type"] in ("tool", "retriever")
            or node["name"] in EXPECTED_TOOL_NAMES
        ):
            tool_spans.append(node)
        elif node["run_type"] in ("llm", "chain"):
            agent_spans.append(node)

        for child in node.get("children", []):
            visit(child)

    visit(execution_trace)

    return {
        "flattened_trace": flattened_trace,
        "agent_spans": agent_spans,
        "tool_spans": tool_spans,
        "span_lookup": span_lookup,
    }


# ============================================================
# Metric Target Mapping
# ============================================================

def first_span(span_lookup, name):
    """Return the first matching span, or None when the stage is absent."""
    matches = span_lookup.get(name, [])
    return matches[0] if matches else None


def build_metric_targets(execution_trace, trace_data):
    """Map the selected metrics to their relevant artifacts for one trace."""
    span_lookup = trace_data["span_lookup"]

    return {
        "trajectory_match": {
            "expected": EXPECTED_TRAJECTORY,
            "actual": execution_trace,
        },
        "parameter_extraction": {
            "actual": first_span(span_lookup, "Parameter Extraction Agent"),
        },
        "plan_accuracy": {
            "actual": first_span(span_lookup, "Supervisor: Plan & Route"),
        },
        "tool_call_accuracy": {
            "expected": EXPECTED_TOOL_CALLS,
            "actual": trace_data["tool_spans"],
        },
        "summary_accuracy": {
            "actual": (
                span_lookup.get("Research Agent", [])
                + span_lookup.get("Analysis Agent", [])
                + span_lookup.get("Report Writer Agent", [])
            ),
        },
        "goal_accomplishment": {
            "actual": first_span(span_lookup, "Supervisor: Final Review"),
        },
    }


def invoke_llm(system_prompt: str, user_prompt: str) -> str:
    """Invoke Azure OpenAI and require a JSON-object response."""
    response = azure_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


# ============================================================
# Build Evaluation Context
# ============================================================

def to_json(value):
    return json.dumps(value, indent=2, default=str, ensure_ascii=False)


def span_inputs(span):
    return (span or {}).get("inputs", {})


def span_outputs(span):
    return (span or {}).get("outputs", {})


def ordered_trajectory(execution_trace):
    """Return the execution order without the root wrapper span."""
    stages = []

    def visit(node, is_root=False):
        if not is_root:
            stages.append({"name": node["name"], "run_type": node["run_type"]})
        for child in node.get("children", []):
            visit(child)

    visit(execution_trace, is_root=True)
    return stages


def build_evaluation_context(metric_name, target, execution_trace):
    """Build the LLM-judge context for one metric and one trace.
    Updated to better support strict Prompts.py style prompts.
    """
    if metric_name == "parameter_extraction":
        span = target["actual"]
        user_req = json.dumps(span_inputs(span), indent=2, default=str, ensure_ascii=False)
        extracted = json.dumps(span_outputs(span), indent=2, default=str, ensure_ascii=False)
        return f"""INPUT_TEXT:
{user_req}

EXTRACTED_PARAMETERS:
{extracted}

EXPECTED_PARAMETER_SCHEMA:
{{"company_profile": "object", "market": "object", "target_year": "number", "objective": "string", "deliverables": "array"}}
"""

    if metric_name == "trajectory_match":
        return f"""reference_outputs:
{json.dumps(target["expected"], indent=2, default=str, ensure_ascii=False)}

actual_outputs:
{json.dumps(ordered_trajectory(target["actual"]), indent=2, default=str, ensure_ascii=False)}

mode:
strict
"""

    if metric_name == "plan_accuracy":
        span = target["actual"]
        return f"""Extracted Parameters:
{json.dumps(span_inputs(span), indent=2, default=str, ensure_ascii=False)}

Generated Execution Plan:
{json.dumps(span_outputs(span), indent=2, default=str, ensure_ascii=False)}
"""

    if metric_name == "tool_call_accuracy":
        return f"""Expected Tool Calls:
{json.dumps(target["expected"], indent=2, default=str, ensure_ascii=False)}

Actual Tool Calls:
{json.dumps(target["actual"], indent=2, default=str, ensure_ascii=False)}
"""

    if metric_name == "summary_accuracy":
        sections = []
        for span in target["actual"]:
            sections.append(f"""Agent: {span["name"]}
Source Context:
{json.dumps(span_inputs(span), indent=2, default=str, ensure_ascii=False)}

Generated Summary:
{json.dumps(span_outputs(span), indent=2, default=str, ensure_ascii=False)}
""")
        return "\n\n".join(sections) or "No summary-producing spans were found."

    if metric_name == "goal_accomplishment":
        final_review = target["actual"]
        return f"""Original User Request:
{json.dumps(execution_trace.get("inputs", {}), indent=2, default=str, ensure_ascii=False)}

Final Workflow Output:
{json.dumps(execution_trace.get("outputs", {}), indent=2, default=str, ensure_ascii=False)}

Final Review:
{json.dumps({"inputs": span_inputs(final_review), "outputs": span_outputs(final_review)}, indent=2, default=str, ensure_ascii=False)}
"""

    raise ValueError(f"Unsupported metric: {metric_name}")


def invoke_llm(system_prompt: str, user_prompt: str) -> str:
    """Invoke Azure OpenAI and require a JSON-object response."""
    response = azure_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


# ============================================================
# Build Evaluation Context
# ============================================================

def to_json(value):
    return json.dumps(value, indent=2, default=str, ensure_ascii=False)


def span_inputs(span):
    return (span or {}).get("inputs", {})


def span_outputs(span):
    return (span or {}).get("outputs", {})


def ordered_trajectory(execution_trace):
    """Return the execution order without the root wrapper span."""
    stages = []

    def visit(node, is_root=False):
        if not is_root:
            stages.append({"name": node["name"], "run_type": node["run_type"]})
        for child in node.get("children", []):
            visit(child)

    visit(execution_trace, is_root=True)
    return stages


def build_evaluation_context(metric_name, target, execution_trace):
    """Build the LLM-judge context for one metric and one trace."""
    if metric_name == "trajectory_match":
        return f"""
Expected Trajectory
===================
{to_json(target["expected"])}

Actual Trajectory
=================
{to_json(ordered_trajectory(target["actual"]))}
"""

    if metric_name == "parameter_extraction":
        span = target["actual"]
        return f"""
Original User Request
=====================
{to_json(span_inputs(span))}

Extracted Parameters
====================
{to_json(span_outputs(span))}
"""

    if metric_name == "plan_accuracy":
        span = target["actual"]
        return f"""
Extracted Parameters
====================
{to_json(span_inputs(span))}

Generated Execution Plan
========================
{to_json(span_outputs(span))}
"""

    if metric_name == "tool_call_accuracy":
        return f"""
Expected Tool Calls
===================
{to_json(target["expected"])}

Actual Tool Calls
=================
{to_json(target["actual"])}
"""

    if metric_name == "summary_accuracy":
        sections = []
        for span in target["actual"]:
            sections.append(f"""
Agent: {span["name"]}
Source Context:
{to_json(span_inputs(span))}

Generated Summary:
{to_json(span_outputs(span))}
""")
        return "\n\n".join(sections) or "No summary-producing spans were found."

    if metric_name == "goal_accomplishment":
        final_review = target["actual"]
        return f"""
Original User Request
=====================
{to_json(execution_trace.get("inputs", {}))}

Final Workflow Output
=====================
{to_json(execution_trace.get("outputs", {}))}

Final Review Input and Output
=============================
{to_json({"inputs": span_inputs(final_review), "outputs": span_outputs(final_review)})}
"""

    raise ValueError(f"Unsupported metric: {metric_name}")


# ============================================================
# Generic Metric Evaluation
# ============================================================

def parse_evaluation_response(response):
    """Parse a JSON response, tolerating harmless surrounding text/code fences."""
    candidate = (response or "").strip()
    if candidate.startswith("```"):
        candidate = candidate.split("\n", 1)[1] if "\n" in candidate else ""
        if candidate.rstrip().endswith("```"):
            candidate = candidate.rstrip()[:-3].rstrip()

    decoder = json.JSONDecoder()
    first_object = candidate.find("{")
    if first_object < 0:
        raise ValueError("No JSON object found in the model response.")

    result, _ = decoder.raw_decode(candidate[first_object:])
    if not isinstance(result, dict):
        raise ValueError("The model response must be a JSON object.")
    return result


def validate_evaluation_result(result, metric_name):
    """Validate the common score contract used by all metric prompts."""
    result["metric"] = metric_name
    score = result.get("score")

    if isinstance(score, str) and score.strip().isdigit():
        score = int(score.strip())
        result["score"] = score

    if isinstance(score, bool) or not isinstance(score, int) or not 1 <= score <= 10:
        raise ValueError("Missing or invalid score; expected an integer from 1 through 10.")

    result["max_score"] = 10
    return result


def evaluate_metric(metric_name, metric_targets, execution_trace):
    """Run one LLM-based metric for one execution trace."""
    response = invoke_llm(
        system_prompt=PROMPTS[metric_name],
        user_prompt=build_evaluation_context(
            metric_name,
            metric_targets[metric_name],
            execution_trace,
        ),
    )

    try:
        result = validate_evaluation_result(
            parse_evaluation_response(response), metric_name
        )
        traceback = result.get("traceback", [])
        result["traceback"] = traceback if isinstance(traceback, list) else [traceback]
        result["raw_evaluation"] = response
        return result
    except (ValueError, json.JSONDecodeError) as error:
        return {
            "metric": metric_name,
            "score": None,
            "max_score": 10,
            "reasoning": f"Invalid evaluator response: {error}",
            "traceback": [],
            "parse_error": str(error),
            "raw_response": response,
        }


## 9. Evaluate and Export Results

Runs every selected metric for every retrieved trace. Results are tagged with trace and variant metadata, then written to `evaluation_results.json` and `evaluation_results.xlsx`. The final output prints an average score per trace.


In [ ]:
# ============================================================
# Execute Evaluation Pipeline for All Retrieved Traces
# ============================================================

def get_variant_id(trace_name):
    """Map the logger's root-run name to an expected-result variant ID."""
    prefix = "Supervisor Agent — "
    return trace_name.removeprefix(prefix) if trace_name.startswith(prefix) else "baseline"


evaluation_results = []

print("=" * 70)
print("Running Evaluation Pipeline")
print("=" * 70)

for trace_number, (root_run, execution_trace) in enumerate(
    zip(root_runs, execution_traces), start=1
):
    print(f"\nTrace {trace_number}/{len(root_runs)}: {root_run.name} ({root_run.id})")

    trace_data = preprocess_execution_trace(execution_trace)
    metric_targets = build_metric_targets(execution_trace, trace_data)

    for metric in SELECTED_METRICS:
        print(f"  Evaluating: {metric}")
        try:
            result = evaluate_metric(metric, metric_targets, execution_trace)
        except Exception as error:
            result = {
                "metric": metric,
                "score": None,
                "max_score": 10,
                "reasoning": f"Evaluation failed: {error}",
            }

        result.update({
            "variant_id": get_variant_id(root_run.name),
            "trace_id": str(root_run.id),
            "trace_name": root_run.name,
            "trace_start_time": (
                root_run.start_time.isoformat() if root_run.start_time else None
            ),
            "lookback_hours": LOOKBACK_HOURS,
        })
        evaluation_results.append(result)

print("=" * 70)
print("Evaluation Completed")
print("=" * 70)
print(f"Traces evaluated  : {len(root_runs)}")
print(f"Metric results    : {len(evaluation_results)}")


# ============================================================
# Process Evaluation Results
# ============================================================

processed_results = [
    {
        "Variant": result.get("variant_id"),
        "Trace ID": result.get("trace_id"),
        "Trace Name": result.get("trace_name"),
        "Trace Start Time": result.get("trace_start_time"),
        "Metric": result.get("metric"),
        "Score": result.get("score"),
        "Max Score": result.get("max_score"),
        "Reasoning": result.get("reasoning"),
        "Recommendation": result.get("recommendation"),
    }
    for result in evaluation_results
]

print(f"Processed {len(processed_results)} metric result(s).")


# ============================================================
# Export Results to JSON
# ============================================================

JSON_OUTPUT_FILE = "evaluation_results.json"

with open(JSON_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=4, ensure_ascii=False)

print(f"✓ JSON results exported to '{JSON_OUTPUT_FILE}'")

# ============================================================
# Export Results to Excel
# ============================================================

EXCEL_OUTPUT_FILE = "evaluation_results.xlsx"

df = pd.DataFrame(processed_results)
df.to_excel(EXCEL_OUTPUT_FILE, index=False)

print(f"✓ Excel results exported to '{EXCEL_OUTPUT_FILE}'")


# ============================================================
# Evaluation Summary
# ============================================================

print("=" * 70)
print("LangSmith Trace Evaluation Completed")
print("=" * 70)
print(f"Lookback window   : {LOOKBACK_HOURS} hour(s)")
print(f"Traces evaluated  : {len(root_runs)}")
print(f"Metric results    : {len(evaluation_results)}")
print(f"JSON Report       : {JSON_OUTPUT_FILE}")
print(f"Excel Report      : {EXCEL_OUTPUT_FILE}")
print("=" * 70)

summary_df = pd.DataFrame(evaluation_results)
summary_df["score"] = pd.to_numeric(summary_df["score"], errors="coerce")

print("\nAverage score by trace:")
for trace_name, group in summary_df.groupby("trace_name", dropna=False):
    average = group["score"].mean()
    score_text = f"{average:.2f}/10" if pd.notna(average) else "N/A"
    print(f"- {trace_name}: {score_text}")

print("\nEvaluation completed successfully.")


Running Evaluation Pipeline

Trace 1/5: Supervisor Agent (019f691b-6b77-7370-84f9-7c60fb1086cf)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 2/5: Supervisor Agent — variant_A_poor_parameter_and_plan (019f691b-6ddd-7bc3-8b6e-9ec2b713c6ef)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 3/5: Supervisor Agent — variant_B_poor_trajectory_and_tools (019f691b-6ee2-7802-a3a7-ea877d9592cf)
  Evaluating: trajectory_match
  Evaluating: parameter_extraction
  Evaluating: plan_accuracy
  Evaluating: tool_call_accuracy
  Evaluating: summary_accuracy
  Evaluating: goal_accomplishment

Trace 4/5: Supervisor Agent — variant_C_poor_summary_accuracy (019f691b-6fd3-7ab0-af61-3ef0c7ff52f5)
  Evaluating: tr

## 10. Interactive Metric Details Report

Creates a standalone HTML report after evaluation. Click any metric score to open its metric-level audit evidence: parameter checks, stage comparisons, plan fragments, tool checks, summary claim evidence, or goal-requirement matches.


In [ ]:
# ============================================================
# Interactive Metric Details Report (Pivoted: 1 Trace = 1 Row)
# ============================================================

from html import escape
from pathlib import Path
from collections import defaultdict
import json


def render_detail_table(items):
    """Render rich traceback details nicely."""
    if not items:
        return "<p class='empty'>No structured details available.</p>"

    if isinstance(items, dict):
        items = [items]

    columns = []
    for item in items:
        if isinstance(item, dict):
            for k in item:
                if k not in columns:
                    columns.append(k)

    if not columns:
        return f"<pre>{escape(json.dumps(items, indent=2, ensure_ascii=False, default=str))}</pre>"

    header = "".join(f"<th>{escape(str(k).replace('_', ' ').title())}</th>" for k in columns)
    rows = []
    for item in items:
        if not isinstance(item, dict):
            item = {"value": str(item)}
        cells = []
        for col in columns:
            val = item.get(col, "—")
            if isinstance(val, (dict, list)):
                val = json.dumps(val, ensure_ascii=False, default=str, indent=2)
            cells.append(f"<td><pre style='white-space:pre-wrap;margin:0'>{escape(str(val))}</pre></td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")

    return f"""<div class="detail-scroll"><table><thead><tr>{header}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>"""


def build_interactive_report(results, output_file="evaluation_details_report.html"):
    """Pivoted report: One row per trace, columns for each metric score."""
    traces = defaultdict(list)
    for r in results:
        key = (r.get("variant_id", "baseline"), r.get("trace_name", "—"), r.get("trace_id", ""))
        traces[key].append(r)

    metric_order = [
        "trajectory_match",
        "parameter_extraction",
        "plan_accuracy",
        "tool_call_accuracy",
        "summary_accuracy",
        "goal_accomplishment",
    ]

    modal_html = []
    table_rows = []
    modal_counter = [0]

    def make_modal(res, variant, trace_name):
        mid = f"modal-{modal_counter[0]}"
        modal_counter[0] += 1
        sc = res.get("score")
        sl = f"{sc}/10" if isinstance(sc, int) else "N/A"
        det = res.get("traceback", [])
        rw = res.get("raw_response") or res.get("raw_evaluation")
        rw_html = (
            f"<details><summary>Raw evaluator response</summary><pre>{escape(str(rw))}</pre></details>"
            if rw else ""
        )

        modal_html.append(f'''<div class="modal-overlay" id="{mid}">
<div class="modal"><div class="modal-header"><h2>{escape(str(res.get("metric","")).replace("_"," ").title())}</h2>
<button onclick="closeModal('{mid}')">&times;</button></div>
<div class="meta"><b>Variant:</b> {escape(str(variant))} | <b>Trace:</b> {escape(str(trace_name))} | <b>Score:</b> {sl}</div>
<h3>Reasoning</h3><p>{escape(str(res.get("reasoning","—")))}</p>
<h3>Details / Traceback</h3>{render_detail_table(det if isinstance(det, list) else [det])}
{rw_html}</div></div>''')
        return mid, sl

    for (var, tname, tid), mres in traces.items():
        cells = []
        for m in metric_order:
            r = next((x for x in mres if x.get("metric") == m), None)
            if r:
                mid, sl = make_modal(r, var, tname)
                sc = r.get("score")
                cls = "high" if isinstance(sc, int) and sc >= 8 else "medium" if isinstance(sc, int) and sc >= 5 else "low"
                cells.append(f'<td><button class="score {cls}" onclick="openModal(\'{mid}\')">{sl}</button></td>')
            else:
                cells.append('<td class="na">N/A</td>')
        scs = [x.get("score") for x in mres if isinstance(x.get("score"), int)]
        avg = round(sum(scs) / len(scs), 2) if scs else "N/A"
        acls = "high" if isinstance(avg, (int, float)) and avg >= 8 else "medium" if isinstance(avg, (int, float)) and avg >= 5 else "low"
        table_rows.append(
            f"<tr><td>{escape(str(var))}</td><td>{escape(str(tname))}</td>{''.join(cells)}<td><span class=\"avg {acls}\">{avg}</span></td></tr>"
        )

    mheads = "".join(f"<th>{m.replace('_',' ').title()}</th>" for m in metric_order)
    doc = f'''<!doctype html>
<html><head><meta charset="utf-8"><title>LangSmith Evaluation Report</title>
<style>
body{{font-family:Inter,Arial,sans-serif;background:#0f172a;color:#e2e8f0;padding:32px}}
table{{border-collapse:collapse;width:100%;background:#1e2937}}
th,td{{padding:11px 13px;border-bottom:1px solid #334155}}
th{{background:#334155;color:#bae6fd}}
.score{{border:0;border-radius:5px;padding:5px 11px;font-weight:700;color:#0f172a;cursor:pointer}}
.high{{background:#4ade80}}.medium{{background:#fbbf24}}.low{{background:#f87171}}
.na{{color:#64748b;font-style:italic}}
.avg{{font-weight:700;padding:4px 10px;border-radius:4px}}
.modal-overlay{{display:none;position:fixed;inset:0;background:rgba(15,23,42,.85);padding:40px;overflow:auto}}
.modal-overlay.open{{display:block}}
.modal{{max-width:1080px;margin:auto;background:#1e2937;border:1px solid #475569;border-radius:10px;padding:22px}}
.modal-header{{display:flex;justify-content:space-between}}
.modal-header button{{font-size:26px;background:none;border:0;color:#e2e8f0;cursor:pointer}}
.detail-scroll{{overflow:auto;max-height:460px}}
.detail-scroll table{{min-width:760px}}
pre{{white-space:pre-wrap;word-break:break-word;background:#0c1322;padding:12px;border-radius:6px}}
.empty{{color:#9ca3af}}
</style></head><body>
<h1>LangSmith Evaluation Report</h1>
<p style="color:#64748b">One row per trace • Click any score to see detailed traceback</p>
<table>
<thead><tr><th>Variant</th><th>Trace</th>{mheads}<th>Avg</th></tr></thead>
<tbody>{''.join(table_rows)}</tbody>
</table>
{''.join(modal_html)}
<script>
function openModal(id){{document.getElementById(id).classList.add("open")}}
function closeModal(id){{document.getElementById(id).classList.remove("open")}}
document.addEventListener("click",e=>{{if(e.target.classList.contains("modal-overlay"))e.target.classList.remove("open")}});
document.addEventListener("keydown",e=>{{if(e.key==="Escape")document.querySelectorAll(".modal-overlay.open").forEach(el=>el.classList.remove("open"))}});
</script></body></html>'''

    Path(output_file).write_text(doc, encoding="utf-8")
    return output_file


HTML_REPORT_FILE = build_interactive_report(evaluation_results)
print(f"✓ Interactive details report exported to '{HTML_REPORT_FILE}'")

✓ Interactive details report exported to 'evaluation_details_report.html'


## 11. Log Evaluation Results back to LangSmith

Logs the evaluation results as feedback into LangSmith.
- **Trace-level metrics** (`goal_accomplishment`, `trajectory_match`) → attached to root trace
- **Span-level metrics** (`parameter_extraction`, `plan_accuracy`, `tool_call_accuracy`, `summary_accuracy`) → attached to their respective child spans


In [ ]:
# ============================================================
# Log Evaluation Results back to LangSmith
# ============================================================

from langsmith import Client

def get_span_id_by_name(execution_trace, span_name):
    """Recursively find the first span with the given name and return its id."""
    if execution_trace.get("name") == span_name:
        return execution_trace.get("id")
    for child in execution_trace.get("children", []):
        found = get_span_id_by_name(child, span_name)
        if found:
            return found
    return None


def log_evaluation_results_to_langsmith(evaluation_results, root_runs, execution_traces):
    """Log evaluation metrics back to LangSmith as feedback."""
    print("=" * 70)
    print("Logging Evaluation Results to LangSmith")
    print("=" * 70)

    # Group results by trace
    results_by_trace = {}
    for res in evaluation_results:
        tid = res.get("trace_id")
        if tid not in results_by_trace:
            results_by_trace[tid] = []
        results_by_trace[tid].append(res)

    logged_count = 0

    for idx, (root_run, exec_trace) in enumerate(zip(root_runs, execution_traces)):
        trace_id = str(root_run.id)
        trace_results = results_by_trace.get(trace_id, [])

        if not trace_results:
            continue

        print(f"\nLogging feedback for trace {idx+1}: {root_run.name}")

        for result in trace_results:
            metric = result.get("metric")
            score = result.get("score")
            reasoning = result.get("reasoning", "")

            if score is None:
                continue

            # Normalize score to 0-1 range for LangSmith (optional but common)
            normalized_score = round(score / 10.0, 4)

            # ============================================================
            # TRACE-LEVEL METRICS → attach to root
            # ============================================================
            if metric in ["goal_accomplishment", "trajectory_match"]:
                try:
                    langsmith_client.create_feedback(
                        key=metric,
                        score=normalized_score,
                        trace_id=trace_id,
                        comment=reasoning[:500] if reasoning else None,
                        source_info={
                            "evaluator": "llm-judge",
                            "raw_score": score,
                            "max_score": 10
                        }
                    )
                    print(f"  ✓ Trace-level: {metric} = {score}/10 → root trace")
                    logged_count += 1
                except Exception as e:
                    print(f"  ✗ Failed to log {metric}: {e}")

            # ============================================================
            # SPAN-LEVEL METRICS → attach to specific spans
            # ============================================================
            elif metric == "parameter_extraction":
                span_id = get_span_id_by_name(exec_trace, "Parameter Extraction Agent")
                if span_id:
                    try:
                        langsmith_client.create_feedback(
                            key="parameter_extraction_accuracy",
                            score=normalized_score,
                            run_id=span_id,
                            trace_id=trace_id,
                            comment=reasoning[:500] if reasoning else None,
                            source_info={"evaluator": "llm-judge", "raw_score": score}
                        )
                        print(f"  ✓ Span-level: parameter_extraction → Parameter Extraction Agent")
                        logged_count += 1
                    except Exception as e:
                        print(f"  ✗ Failed: {e}")

            elif metric == "plan_accuracy":
                span_id = get_span_id_by_name(exec_trace, "Supervisor: Plan & Route")
                if span_id:
                    try:
                        langsmith_client.create_feedback(
                            key="plan_accuracy",
                            score=normalized_score,
                            run_id=span_id,
                            trace_id=trace_id,
                            comment=reasoning[:500] if reasoning else None,
                            source_info={"evaluator": "llm-judge", "raw_score": score}
                        )
                        print(f"  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route")
                        logged_count += 1
                    except Exception as e:
                        print(f"  ✗ Failed: {e}")

            elif metric == "tool_call_accuracy":
                # Attach to Research Agent and Analysis Agent as examples
                for agent_name in ["Research Agent", "Analysis Agent", "Report Writer Agent"]:
                    span_id = get_span_id_by_name(exec_trace, agent_name)
                    if span_id:
                        try:
                            langsmith_client.create_feedback(
                                key=f"tool_call_accuracy_{agent_name.replace(' ', '_').lower()}",
                                score=normalized_score,
                                run_id=span_id,
                                trace_id=trace_id,
                                comment=reasoning[:400] if reasoning else None,
                                source_info={"evaluator": "llm-judge", "raw_score": score}
                            )
                            print(f"  ✓ Span-level: tool_call_accuracy → {agent_name}")
                            logged_count += 1
                        except Exception as e:
                            print(f"  ✗ Failed for {agent_name}: {e}")

            elif metric == "summary_accuracy":
                for agent_name in ["Research Agent", "Analysis Agent", "Report Writer Agent"]:
                    span_id = get_span_id_by_name(exec_trace, agent_name)
                    if span_id:
                        try:
                            langsmith_client.create_feedback(
                                key=f"summary_accuracy_{agent_name.replace(' ', '_').lower()}",
                                score=normalized_score,
                                run_id=span_id,
                                trace_id=trace_id,
                                comment=reasoning[:400] if reasoning else None,
                                source_info={"evaluator": "llm-judge", "raw_score": score}
                            )
                            print(f"  ✓ Span-level: summary_accuracy → {agent_name}")
                            logged_count += 1
                        except Exception as e:
                            print(f"  ✗ Failed for {agent_name}: {e}")

    print("=" * 70)
    print(f"Successfully logged {logged_count} feedback entries to LangSmith.")
    print("=" * 70)


# Run the logging
log_evaluation_results_to_langsmith(evaluation_results, root_runs, execution_traces)


Logging Evaluation Results to LangSmith

Logging feedback for trace 1: Supervisor Agent
  ✓ Trace-level: trajectory_match = 8/10 → root trace
  ✓ Span-level: parameter_extraction → Parameter Extraction Agent
  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route
  ✓ Span-level: tool_call_accuracy → Research Agent
  ✓ Span-level: tool_call_accuracy → Analysis Agent
  ✓ Span-level: tool_call_accuracy → Report Writer Agent
  ✓ Span-level: summary_accuracy → Research Agent
  ✓ Span-level: summary_accuracy → Analysis Agent
  ✓ Span-level: summary_accuracy → Report Writer Agent
  ✓ Trace-level: goal_accomplishment = 4/10 → root trace

Logging feedback for trace 2: Supervisor Agent — variant_A_poor_parameter_and_plan
  ✓ Trace-level: trajectory_match = 8/10 → root trace
  ✓ Span-level: parameter_extraction → Parameter Extraction Agent
  ✓ Span-level: plan_accuracy → Supervisor: Plan & Route
  ✓ Span-level: tool_call_accuracy → Research Agent
  ✓ Span-level: tool_call_accuracy → Analysis Age